# DeepEval Agent Evaluation — 4 Agent Metrics

This notebook runs `weather_agent` (`../weather_agent/agent.py`) — a real agent that calls a live
public weather API (open-meteo.com, no key needed) — through DeepEval's 4 agent-testing metrics.
Everything below runs against the live agent: real HTTP calls, real (and constantly changing)
weather data, real LLM-judged scores. Nothing here is a mocked or hand-authored test case.

**Why bother testing an agent's tool use at all?** In February 2024, Air Canada's support chatbot
told a grieving customer that a bereavement discount could be claimed *after* travel — a policy
that didn't exist. The airline was held liable by Canada's Civil Resolution Tribunal. The root
cause: the bot answered a policy question from its own (wrong) memory instead of calling the
authoritative lookup tool. That's the exact failure mode `ToolCorrectnessMetric` below exists to
catch before it reaches a real user.

The 4 sections below explain each metric in detail: what it checks, exactly how it's computed, and
what would make it fail. Then the rest of the notebook runs all 4 live against 8 real questions.

## Metric 1 — `TaskCompletionMetric`

**What it checks:** did the agent actually satisfy the user's underlying goal — not just "did it
say something plausible", but "was the task genuinely completed".

**How it's computed:** you give the metric a `task` string when you construct it — a description
of what the agent is *for* overall (e.g. "answer weather questions by calling the appropriate live
tool"), not the specific question being asked in any one test case. At `measure()` time, the LLM
judge is given: the `task` description, the test case's `input` (the specific question), the
`actual_output` (what the agent said), and — if present — the execution trace of what the agent
did to get there. The judge reasons in natural language about whether the goal was achieved and
returns a score from 0 to 1 plus a written justification (`metric.reason`).

**What causes a low score:**
- The agent gave an evasive, incomplete, or refused answer when it could have helped.
- The agent's output doesn't actually address what was asked, even if it sounds confident.
- A `task` description that's broader than any single query could satisfy — e.g. describing the
  agent as answering "current conditions **and** forecasts" will cause the judge to partially
  penalize a purely-current-conditions answer for not *also* covering forecasts. This is a real
  trap: the `task` string should describe the agent's overall competence, not an implicit
  checklist every single answer must fully cover.

## Metric 2 — `ToolCorrectnessMetric`

**What it checks:** was the *right* tool called — or was none called at all — against a ground
truth **you** declare per test case.

**How it's computed:** unlike the other 3 metrics, this one is primarily **structural/deterministic**,
not an LLM judgment call. You declare `expected_tools` on the `LLMTestCase` (a list of `ToolCall`
objects naming which tool(s) *should* have been used). At `measure()` time, it compares the agent's
real `tools_called` against `expected_tools`:
- By **default**, it only compares tool **names** and completely ignores arguments — two tool
  calls with the same name but wildly different arguments still count as a match.
- Passing `evaluation_params=[ToolCallParams.INPUT_PARAMETERS]` (used in this notebook) makes it
  also compare each expected tool's `input_parameters` dict against the actual call, key by key —
  turning it into a real, deterministic argument check, not just a tool-selection check.
- `should_exact_match=True` requires the called-tools set to match `expected_tools` exactly (no
  extra, unexpected tool calls allowed).
- `should_consider_ordering=True` additionally checks the tools were called in the expected
  sequence.
- If you also pass `available_tools` (the full list the agent *could* have chosen from), it layers
  on an additional LLM-judged "was this the best tool among the available options" check.

**What causes a low score:** the wrong tool was called, no tool was called when one was required
(the Air Canada pattern), or — with `ToolCallParams.INPUT_PARAMETERS` enabled — the right tool was
called with the wrong arguments (e.g. the wrong city).

## Metric 3 — `ArgumentCorrectnessMetric`

**What it checks:** right tool, but were the *arguments* passed to it correct for the specific
question asked.

**How it's computed:** pure LLM judgment, with **no ground truth field at all**. Checked directly
against the installed DeepEval source: the constructor takes only `threshold`, `model`,
`include_reason`, `async_mode`, `strict_mode`, `verbose_mode` — there is nowhere to declare
"the correct arguments were X". At `measure()` time, the judge is given only the natural-language
`input` and the actual `tools_called` (with their real arguments), and reasons about whether those
arguments make sense for that question. It is inferring correctness from the question text alone,
not comparing against anything you wrote down.

**What causes a low score:** an argument clearly wrong for the question (e.g. a city name copied
from earlier context instead of the one just asked about) — or, just as easily, **the judge's own
reasoning being wrong**. In an earlier run of this notebook, asking for "tomorrow's forecast" was
scored as a wrong-argument case, because the judge assumed "tomorrow" means exactly 1 day of data,
when the tool's `days` parameter actually counts from *today* (so `days=2` is what correctly
reaches tomorrow). That wasn't a bug in the agent — it's a real limitation of judging arguments
from natural language alone, with no declared ground truth to check against. This is exactly why
`ToolCorrectnessMetric` + `ToolCallParams.INPUT_PARAMETERS` (Metric 2) matters as a deterministic
cross-check: it catches the same failure mode without depending on the judge's semantic reasoning
being right. The golden cases in this notebook are phrased with explicit day counts (e.g. "a 3-day
forecast") specifically to avoid this kind of ambiguity.

## Metric 4 — `StepEfficiencyMetric`

**What it checks:** did the agent reach its answer without unnecessary or redundant tool calls —
the tool-calling analog of an "infinite retrieval loop" check.

**How it's computed:** the judge is given an **execution trace** — a structured record of which
tool(s) were called, in what order — and first extracts what the underlying task was, then reasons
about whether every step in the trace was actually necessary to complete it.

**Important implementation detail:** this metric reads `test_case._trace_dict`, a private field
that is *not* populated automatically from `tools_called`. In typical DeepEval usage it comes from
wrapping your tools with `deepeval.tracing.observe()` decorators, which record a full span tree as
your agent runs. Since `weather_agent`'s tools are plain functions with no DeepEval tracing wired
in, this notebook builds the equivalent trace by hand: `attach_trace()` (in
`weather_agent/agent.py`) constructs a minimal `BaseSpan`/`ToolSpan` tree directly from the same
`ToolCallRecord` data `to_deepeval_tool_calls()` already captures, and hands DeepEval's own
`trace_manager.create_nested_spans_dict()` that tree to serialize. Without this, the metric reports
"execution trace is null" and cannot score anything.

**What causes a low score:** calling more tools than the question required (e.g. checking current
conditions *and* a forecast when only one was asked about), speculative "just in case" calls, or
calling a tool, then calling it again with corrected arguments (the first call was wasted).

## Setup

In [1]:
import json
import subprocess
import sys
from pathlib import Path

WEATHER_AGENT_DIR = Path("..", "weather_agent").resolve()
sys.path.insert(0, str(WEATHER_AGENT_DIR))

from agent import ToolCallRecord, AgentResponse, to_deepeval_tool_calls, attach_trace

from deepeval.test_case import LLMTestCase, ToolCall, ToolCallParams
from deepeval.metrics import (
    TaskCompletionMetric,
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,
    StepEfficiencyMetric,
)
from deepeval.metrics.utils import should_use_azure_openai

TASK_DESCRIPTION = (
    "Answer the user's specific weather question -- current conditions or a forecast, "
    "whichever was asked -- for a real place, by calling the appropriate live weather tool."
)


def run_agent_live(query: str, verbose: bool = True) -> AgentResponse:
    """Run weather_agent in a fresh subprocess (avoids an ipykernel/nest_asyncio
    conflict that can otherwise break the agent's underlying HTTP client on retry)."""
    result = subprocess.run(
        [sys.executable, str(WEATHER_AGENT_DIR / "cli.py"), query],
        capture_output=True, text=True, timeout=120,
    )
    if result.returncode != 0:
        raise RuntimeError(f"weather_agent subprocess failed:\n{result.stderr}")
    data = json.loads(result.stdout)
    tools_called = [ToolCallRecord(**tc) for tc in data["tools_called"]]
    if verbose:
        for tc in tools_called:
            print(f"[tool call] {tc.name}({tc.input_parameters})")
    return AgentResponse(output=data["output"], tools_called=tools_called)


print(f"DeepEval judge is Azure OpenAI: {should_use_azure_openai()}")

DeepEval judge is Azure OpenAI: True


## Golden cases

Eight real questions, covering both tools across different cities and day counts, phrased so the
"right" tool and arguments are unambiguous. Each case declares its own `expected_tool` **and**
`expected_input_parameters` — the actual ground truth this notebook checks the live agent against:

| Query | Expected tool | Expected arguments |
|---|---|---|
| Current weather in Tokyo | `get_current_weather` | `city='Tokyo'` |
| Current temperature in Bengaluru | `get_current_weather` | `city='Bengaluru'` |
| Current weather in New York | `get_current_weather` | `city='New York'` |
| Current temperature in Mumbai | `get_current_weather` | `city='Mumbai'` |
| 3-day forecast for Paris | `get_forecast` | `city='Paris', days=3` |
| 5-day forecast for Berlin | `get_forecast` | `city='Berlin', days=5` |
| 7-day forecast for London | `get_forecast` | `city='London', days=7` |
| 2-day forecast for Sydney | `get_forecast` | `city='Sydney', days=2` |

Explicit day counts (rather than "tomorrow"-style phrasing) keep the arguments unambiguous for
both the deterministic check (Metric 2) and the LLM-judged one (Metric 3) — see the caveat above.

In [ ]:
cases = [
    {
        "query": "What is the current weather in Tokyo?",
        "expected_tool": "get_current_weather",
        "expected_input_parameters": {"city": "Tokyo"},
    },
    {
        "query": "What is the current temperature in Bengaluru?",
        "expected_tool": "get_current_weather",
        "expected_input_parameters": {"city": "Bengaluru"},
    },
    {
        "query": "What is the current weather in New York?",
        "expected_tool": "get_current_weather",
        "expected_input_parameters": {"city": "New York"},
    },
    {
        "query": "What is the current temperature in Mumbai?",
        "expected_tool": "get_current_weather",
        "expected_input_parameters": {"city": "Mumbai"},
    },
    {
        "query": "Give me a 3-day weather forecast for Paris.",
        "expected_tool": "get_forecast",
        "expected_input_parameters": {"city": "Paris", "days": 3},
    },
    {
        "query": "What will the weather be like in Berlin over the next 5 days?",
        "expected_tool": "get_forecast",
        "expected_input_parameters": {"city": "Berlin", "days": 5},
    },
    {
        "query": "Give me a 7-day weather forecast for London.",
        "expected_tool": "get_forecast",
        "expected_input_parameters": {"city": "London", "days": 7},
    },
    {
        "query": "What will the weather be like in Sydney over the next 2 days?",
        "expected_tool": "get_forecast",
        "expected_input_parameters": {"city": "Sydney", "days": 2},
    },
]

for case in cases:
    print(f"--- {case['query']}")
    response = run_agent_live(case["query"])
    case["response"] = response
    print(f"Agent output: {response.output}\n")

## Run all 4 metrics on each case

`ToolCorrectnessMetric` needs `expected_tools` (the tool -- and, since we also pass
`evaluation_params=[ToolCallParams.INPUT_PARAMETERS]` below, the exact arguments -- you assert
*should* have been used; that assertion is the actual test). By default this metric only compares
tool **names** and ignores arguments entirely; declaring `input_parameters` on `expected_tools` and
enabling `ToolCallParams.INPUT_PARAMETERS` turns it into a deterministic argument check too, as a
cross-check against `ArgumentCorrectnessMetric`'s LLM-judged (ground-truth-free) version.
`StepEfficiencyMetric` needs an execution trace, attached via `attach_trace` (built from the same
tool-call records `to_deepeval_tool_calls` uses).

In [ ]:
results = []

for case in cases:
    response = case["response"]
    test_case = LLMTestCase(
        input=case["query"],
        actual_output=response.output,
        tools_called=to_deepeval_tool_calls(response),
        expected_tools=[
            ToolCall(name=case["expected_tool"], input_parameters=case["expected_input_parameters"])
        ],
    )
    attach_trace(test_case, case["query"], response)

    metrics = [
        ("TaskCompletionMetric", TaskCompletionMetric(task=TASK_DESCRIPTION, threshold=0.7, async_mode=False)),
        (
            "ToolCorrectnessMetric",
            ToolCorrectnessMetric(
                threshold=0.7, async_mode=False, evaluation_params=[ToolCallParams.INPUT_PARAMETERS]
            ),
        ),
        ("ArgumentCorrectnessMetric", ArgumentCorrectnessMetric(threshold=0.7, async_mode=False)),
        ("StepEfficiencyMetric", StepEfficiencyMetric(threshold=0.7, async_mode=False)),
    ]

    print(f"=== {case['query']}")
    row = {"query": case["query"]}
    for name, metric in metrics:
        metric.measure(test_case)
        row[name] = (metric.score, metric.is_successful())
        print(f"  {name:<26} score={metric.score:.2f}  {'PASS' if metric.is_successful() else 'FAIL'}")
        print(f"    reason: {metric.reason}")
    results.append(row)
    print()

## Coverage matrix

Which of the 4 metrics passed, for each live case.

In [ ]:
metric_names = ["TaskCompletionMetric", "ToolCorrectnessMetric", "ArgumentCorrectnessMetric", "StepEfficiencyMetric"]

col_w = 20
header = f"{'Query':<45}|" + "".join(f" {m:<{col_w-1}}|" for m in metric_names)
print(header)
print("-" * 45 + "|" + ("-" * col_w + "|") * len(metric_names))
for row in results:
    line = f"{row['query'][:44]:<45}|"
    for m in metric_names:
        score, passed = row[m]
        cell = f"{'PASS' if passed else 'FAIL'} ({score:.2f})"
        line += f" {cell:<{col_w-1}}|"
    print(line)

print()
print("Per-metric pass rate across all live cases:")
for m in metric_names:
    passed_count = sum(1 for row in results if row[m][1])
    print(f"  {m:<26} {passed_count}/{len(results)} passed")

## Reading the results

- **A metric failing here is a real finding, not a bug in this notebook.** Every score above came
  from an actual live call to weather_agent and an actual LLM judge call — there's no mocked data
  to "fix" if a score looks wrong. If `ArgumentCorrectnessMetric` fails a case that looks correct
  to you, that's the judge's reasoning to inspect (see Metric 3 above), not a broken test.
- **8 cases is still a starting point, not full coverage.** This golden set only covers
  `get_current_weather` and `get_forecast` with clean, well-formed city names. It doesn't yet probe
  an ambiguous city name (e.g. one that exists in multiple countries), a nonexistent place, or a
  question that shouldn't need a tool call at all (a hallucination-pressure test, the Air Canada
  pattern from the intro). Extending `cases` above with those is the natural next step.
- **Why every metric is measured with `async_mode=False`:** this notebook's Jupyter kernel has
  `nest_asyncio` applied (for top-level `await` support), which breaks DeepEval's default async
  loop handling on repeated metric calls. Forcing the synchronous code path sidesteps that
  entirely — it doesn't change what's being measured, just how the judge call is made.